In [14]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [15]:
PROJECT_ROOT = Path(".").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "lendingclub"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOAD_FULL = True
NROWS = None if LOAD_FULL else 50000

In [16]:
def find_file(root: Path, keywords):
    for f in root.rglob("*"):
        if f.is_file():
            name = f.name.lower()
            if all(k.lower() in name for k in keywords):
                return f
    return None

accepted_file = find_file(RAW_DIR, ["accepted"])
rejected_file = find_file(RAW_DIR, ["rejected"])

print("Accepted file:", accepted_file)
print("Rejected file:", rejected_file)

Accepted file: /Users/pattonpatton/Desktop/RiskProject/Risk_Management_Project/ModelingThoughts/data/raw/lendingclub/accepted_2007_to_2018Q4.csv.gz
Rejected file: /Users/pattonpatton/Desktop/RiskProject/Risk_Management_Project/ModelingThoughts/data/raw/lendingclub/rejected_2007_to_2018Q4.csv.gz


In [17]:
if accepted_file is None:
    raise FileNotFoundError("Could not find accepted loans file in data/raw/lendingclub")

suffixes = "".join(accepted_file.suffixes).lower()

if suffixes.endswith(".csv.gz"):
    accepted_df = pd.read_csv(
        accepted_file,
        compression="gzip",
        low_memory=False,
        nrows=NROWS
    )
else:
    accepted_df = pd.read_csv(
        accepted_file,
        low_memory=False,
        nrows=NROWS
    )

accepted_df.columns = accepted_df.columns.str.strip()

print("accepted_df shape:", accepted_df.shape)
accepted_df.head()

accepted_df shape: (2260701, 151)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
accepted_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), str(38)
memory usage: 3.1 GB


In [19]:
accepted_df.head().T.head(40)

,0,1,2,3,4
id,68407277,68355089,68341763,66310712,68476807
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt_inv,3600.0,24700.0,20000.0,35000.0,10400.0
term,36 months,36 months,60 months,60 months,60 months
int_rate,13.99,11.99,10.78,14.85,22.45
installment,123.03,820.28,432.66,829.9,289.91
grade,C,C,B,C,F
sub_grade,C4,C1,B4,C5,F1


In [20]:
accepted_df["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [21]:
missing = accepted_df.isnull().mean().sort_values(ascending=False)
missing.head(30)

member_id                                     1.000000
orig_projected_additional_accrued_interest    0.996173
hardship_end_date                             0.995171
hardship_start_date                           0.995171
hardship_type                                 0.995171
hardship_reason                               0.995171
hardship_status                               0.995171
deferral_term                                 0.995171
hardship_last_payment_amount                  0.995171
hardship_payoff_balance_amount                0.995171
hardship_loan_status                          0.995171
hardship_dpd                                  0.995171
hardship_length                               0.995171
payment_plan_start_date                       0.995171
hardship_amount                               0.995171
settlement_term                               0.984852
debt_settlement_flag_date                     0.984852
settlement_status                             0.984852
settlement

**Current feature choices**

In [22]:
# Start from raw accepted data
df = accepted_df.copy()

# Parse dates first
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")

# Clean term
df["term"] = (
    df["term"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

# Clean revol_util
df["revol_util"] = (
    df["revol_util"]
    .astype(str)
    .str.replace("%", "", regex=False)
)
df["revol_util"] = pd.to_numeric(df["revol_util"], errors="coerce")

# Create engineered columns
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["credit_history_years"] = (
    (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25
)

# Now choose final modeling columns
feature_cols = [
    "loan_amnt",
    "term",
    "installment",
    # "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies",
    "fico_avg",
    "credit_history_years",
    # "int_rate",
]

df = df[feature_cols + ["loan_status"]].copy()
print(df.shape)
df.head()

(2260701, 23)


,loan_amnt,term,installment,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies,fico_avg,credit_history_years,loan_status
0,3600.0,36.0,123.03,10+ years,MORTGAGE,55000.0,Not Verified,2015-12-01,debt_consolidation,PA,...,7.0,0.0,2765.0,29.7,13.0,1.0,0.0,677.0,12.334018,Fully Paid
1,24700.0,36.0,820.28,10+ years,MORTGAGE,65000.0,Not Verified,2015-12-01,small_business,SD,...,22.0,0.0,21470.0,19.2,38.0,4.0,0.0,717.0,16.000000,Fully Paid
2,20000.0,60.0,432.66,10+ years,MORTGAGE,63000.0,Not Verified,2015-12-01,home_improvement,IL,...,6.0,0.0,7869.0,56.2,18.0,5.0,0.0,697.0,15.331964,Fully Paid
3,35000.0,60.0,829.90,10+ years,MORTGAGE,110000.0,Source Verified,2015-12-01,debt_consolidation,NJ,...,13.0,0.0,7802.0,11.6,17.0,1.0,0.0,787.0,7.247091,Current
4,10400.0,60.0,289.91,3 years,MORTGAGE,104433.0,Source Verified,2015-12-01,major_purchase,PA,...,12.0,0.0,21929.0,64.5,35.0,6.0,0.0,697.0,17.500342,Fully Paid


In [23]:
accepted_df["issue_d"] = pd.to_datetime(accepted_df["issue_d"], format="%b-%Y", errors="coerce")
accepted_df["earliest_cr_line"] = pd.to_datetime(accepted_df["earliest_cr_line"], format="%b-%Y", errors="coerce")
accepted_df["fico_avg"] = (accepted_df["fico_range_low"] + accepted_df["fico_range_high"]) / 2
accepted_df["credit_history_years"] = (
    (accepted_df["issue_d"] - accepted_df["earliest_cr_line"]).dt.days / 365.25
)

In [24]:
# ---------------------------------
# Define target classes
# ---------------------------------
bad_statuses = {
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)"
}

good_statuses = {
    "Fully Paid"
}

In [25]:
df_model = df[df["loan_status"].isin(bad_statuses | good_statuses)].copy()
df_model["target_bad"] = df_model["loan_status"].isin(bad_statuses).astype(int)

print("df_model shape:", df_model.shape)
print(df_model["target_bad"].value_counts(dropna=False))
print("Bad rate:", df_model["target_bad"].mean())

df_model shape: (1371166, 24)
target_bad
0    1076751
1     294415
Name: count, dtype: int64
Bad rate: 0.21471871385375657


In [26]:
from pathlib import Path

out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

df_model.to_csv(out_dir / "accepted_model_base.csv", index=False)
print("saved")

saved
